# Modeling Baselines


In this baseline modeling process, we perform the following sequence of steps:

1. We first preprocess the data. 

2. Impute missing values using the most frequent value for categorical variables and the median for numerical variables.

3. Numerical columns are then rescaled so they are distributed like N(0,1).

4. Perform a stratified split that preserves the grouping by patient_nbr since patients can have repeat visits to the hospital.

5. Fit a logistic regression model through the above pipeline and the use it to make predictions. 

6. Evaluate the performance of the model using various metrics. Recall from our EDA that our target metric is F1-Score.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_recall_curve, 
                             precision_score, recall_score, f1_score, confusion_matrix, 
                             log_loss, average_precision_score, classification_report)

from scoring import score

pd.set_option('display.max_columns', None)

In [2]:
from preprocessing import preprocess_data

In [3]:
df = pd.read_csv('../data/diabetic_data.csv')

display(df.head())
print(df.columns)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')


In [4]:
# Data preprocessing based off of previous EDA

from preprocessing import preprocess_data

# Extract numerical and categorical columns

def num_cat(df,target):
    # Identify numerical and categorical columns
    df = df.drop(columns=[target])
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    return num_cols, cat_cols

In [5]:
df = preprocess_data(df)

In [6]:
df["admission_type_id"]

0              NULL
1         Emergency
2         Emergency
3         Emergency
4         Emergency
            ...    
101761    Emergency
101762    Emergency
101763    Emergency
101764       Urgent
101765    Emergency
Name: admission_type_id, Length: 101766, dtype: str

In [7]:
num_cols, cat_cols = num_cat(df,"target")
print(num_cols)
print(cat_cols)
features = num_cols+cat_cols

X = df[features]
y = df['target'].astype(int) 

['patient_nbr', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
['race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'a1cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesmed']


/var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_7163/2228909507.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns.tolist()


In [8]:
# The target is imbalanced. We also want to keep repeat patients together.

# 1. Set n_splits=5 so the test size naturally becomes 1/5 = 20%
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Get the indices for the very first split next() gets only the first split for us
train_idx, test_idx = next(sgkf.split(X, y, groups=X['patient_nbr']))

# 3. Create a single train/test split
Xt, Xv = X.iloc[train_idx], X.iloc[test_idx]
yt, yv = y.iloc[train_idx], y.iloc[test_idx]

In [9]:
# We will save the train / test split here and use it for the modeling experiments

train = Xt.copy()
train["target"] = yt
train.to_csv("../data/cleaned_data/train.csv",index=False)

test = Xv.copy()
test["target"] = yv 
test.to_csv("../data/cleaned_data/test.csv",index=False)

train.drop(columns=['patient_nbr'])
test.drop(columns=['patient_nbr'])

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,payer_code,medical_specialty,diag_1,diag_2,diag_3,max_glu_serum,a1cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesmed,target
9,12,33,3,18,0,0,0,8,Caucasian,Female,[90-100),NaN,Elective,Discharged/transferred to SNF,Transfer from a hospital,NaN,InternalMedicine,434,198,486,NaN,NaN,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,0
12,7,60,0,15,0,1,0,8,Caucasian,Female,[40-50),NaN,Emergency,Discharged/transferred to SNF,Emergency Room,NaN,Family/GeneralPractice,428,250.43,250.6,NaN,NaN,Steady,Up,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,1
13,10,55,1,31,0,0,0,8,Caucasian,Male,[80-90),NaN,Emergency,Discharged/transferred to home with home healt...,Emergency Room,NaN,Family/GeneralPractice,428,411,427,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,0
15,12,75,5,13,0,0,0,9,AfricanAmerican,Male,[60-70),NaN,Emergency,Discharged/transferred to SNF,Emergency Room,NaN,NaN,999,507,996,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,0
21,2,36,2,11,0,0,0,6,NaN,Male,[60-70),NaN,Urgent,Discharged to home,Transfer from a hospital,NaN,NaN,572,456,427,NaN,NaN,Steady,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101729,4,35,0,14,0,0,0,9,Caucasian,Female,[70-80),NaN,Elective,Discharged to home,Physician Referral,MC,NaN,276,466,401,NaN,NaN,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,0
101736,8,54,1,20,1,0,1,9,Caucasian,Female,[60-70),NaN,Emergency,Discharged/transferred to SNF,Emergency Room,HM,NaN,590,564,250.8,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,0
101741,4,21,1,8,0,0,1,9,AfricanAmerican,Male,[90-100),NaN,Emergency,Discharged/transferred to SNF,Emergency Room,MC,NaN,820,427,276,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,0
101753,13,51,2,13,0,0,0,9,Other,Male,[40-50),NaN,Emergency,Discharged to home,Emergency Room,NaN,NaN,250.8,730,731,NaN,NaN,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,0


In [10]:
# We will make a Pipeline for the Baseline Model with a Preprocessing Step and a Logistic Regression Model.
# The preprocessing should only be done using the training data.

# A naive baseline will be to simply use a median and most frequent imputer for the numerical and categorical columns respectively.

num = Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))])
prep = ColumnTransformer([("num",num,num_cols),("cat",cat,cat_cols)])
clf = Pipeline([("prep", prep),("model", LogisticRegression(max_iter=500, random_state=42))])


In [11]:
# Fit the model to the training data

clf.fit(Xt,yt)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains s

In [12]:
# Have the model now make predictions
labels = clf.predict(Xv)                   
probs = clf.predict_proba(Xv)
proba = probs[:, 1]
score(yv, y_pred=None, proba=proba, threshold=0.5)

Accuracy: 0.887
Confusion matrix:
 [[18011    71]
 [ 2234    38]]
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18082
           1       0.35      0.02      0.03      2272

    accuracy                           0.89     20354
   macro avg       0.62      0.51      0.49     20354
weighted avg       0.83      0.89      0.84     20354

F1 @0.5: 0.032
F2 @0.5: 0.021
ROC-AUC: 0.6632
PR-AUC:  0.2112

Best-threshold F1: 0.27729 at threshold 0.13801.

Best-threshold F2: 0.41416 at threshold 0.07053.


In [13]:
print(f"If we were to simply predict that the discharged patient would not return within")
print(f"30 days no matter what the model would get it correct {100*np.round(1 - sum(yv)/len(yv),3)}% of the time.")

If we were to simply predict that the discharged patient would not return within
30 days no matter what the model would get it correct 88.8% of the time.


# Conclusion

1. Precision is at 0.35 but Recall is at 0.02 which is very low and not good for the model's purpose.

2. The best $F_1$-score is around 0.27729 which is not really good either. Also its best $F_2$-score is $0.41416$. Note that the threshold is lower for $F_2$-score to be optimal. Unfortunately, the model as it stands is not going to do a very good job of fulfilling its mission of helping reduce hospital readmissions.